# Lab 10 - subgroup evaluation, intervals, and a model card

**Session 10.** Take a model that validates well and ask the three closing questions: which
number, how certain, and who bears the error. Finish by completing the model card.

## 1. A model, and a group variable

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X, y = make_classification(n_samples=2000, n_features=10, n_informative=5,
                           class_sep=0.8, random_state=2026)
rng = np.random.default_rng(2026)

# Group B is 20% of the sample, and its signal is deliberately noisier - the ordinary
# situation for an under-represented subgroup, not a contrived one.
group = np.where(rng.random(len(y)) < 0.2, "B", "A")
X[group == "B"] += rng.normal(0, 1.4, X[group == "B"].shape)

X_tr, X_te, y_tr, y_te, g_tr, g_te = train_test_split(
    X, y, group, test_size=0.4, stratify=y, random_state=2026)

model = Pipeline([("sc", StandardScaler()),
                  ("clf", LogisticRegression(max_iter=1000))]).fit(X_tr, y_tr)
p_te = model.predict_proba(X_te)[:, 1]
print(f"test n = {len(y_te)}; group B share = {(g_te == 'B').mean():.2f}; "
      f"base rate = {y_te.mean():.2f}")

## 2. The headline number, with an interval

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score


def bootstrap_ci(y_true, scores, metric, n_boot=2000, level=0.95, seed=0):
    """Percentile bootstrap over the TEST rows."""
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_true), len(y_true))
        if len(np.unique(y_true[idx])) < 2:
            continue
        out.append(metric(y_true[idx], scores[idx]))
    out = np.sort(out)
    lo = out[int((1 - level) / 2 * len(out))]
    hi = out[int((1 + level) / 2 * len(out)) - 1]
    return float(lo), float(hi)


auc = roc_auc_score(y_te, p_te)
lo, hi = bootstrap_ci(y_te, p_te, roc_auc_score)
print(f"AUC = {auc:.3f}   95% CI [{lo:.3f}, {hi:.3f}]   (width {hi - lo:.3f})")
print(f"majority-class baseline accuracy = {max(y_te.mean(), 1 - y_te.mean()):.3f}")
print(f"model accuracy at 0.5           = {accuracy_score(y_te, p_te >= 0.5):.3f}")

## 3. Disaggregate

In [ ]:
print(f"{'group':<6} {'n':>5} {'base':>6} {'acc':>7} {'recall':>8} {'AUC':>7}   95% CI on AUC")
for g in ("A", "B", None):
    m = np.ones(len(y_te), bool) if g is None else (g_te == g)
    yy, pp = y_te[m], p_te[m]
    lo, hi = bootstrap_ci(yy, pp, roc_auc_score, seed=1)
    label = "all" if g is None else g
    print(f"{label:<6} {m.sum():>5} {yy.mean():>6.2f} "
          f"{accuracy_score(yy, pp >= 0.5):>7.3f} {recall_score(yy, pp >= 0.5):>8.3f} "
          f"{roc_auc_score(yy, pp):>7.3f}   [{lo:.3f}, {hi:.3f}]")

rec = {g: recall_score(y_te[g_te == g], p_te[g_te == g] >= 0.5) for g in ("A", "B")}
print(f"\nrecall gap A - B = {rec['A'] - rec['B']:+.3f}")
print("The 'all' row is an average over people. Report the worst group beside it.")

## 4. Calibration, separately from discrimination

A model can rank well and still be systematically overconfident. If the probability feeds a
budget or a threshold, it must be calibrated - and it must be calibrated *per group*.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(figsize=(5.5, 4))
ax.plot([0, 1], [0, 1], "--", color="grey", label="perfect")
for g in ("A", "B"):
    m = g_te == g
    frac, mean_pred = calibration_curve(y_te[m], p_te[m], n_bins=8, strategy="quantile")
    ax.plot(mean_pred, frac, marker="o", label=f"group {g}")
ax.set(xlabel="mean predicted probability", ylabel="observed frequency",
       title="Reliability diagram, by group")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Does a fairness criterion hold?

In [ ]:
from sklearn.metrics import confusion_matrix

print(f"{'group':<6} {'pos rate':>9} {'TPR':>7} {'FPR':>7}")
for g in ("A", "B"):
    m = g_te == g
    pred = p_te[m] >= 0.5
    tn, fp, fn, tp = confusion_matrix(y_te[m], pred).ravel()
    print(f"{g:<6} {pred.mean():>9.3f} {tp / (tp + fn):>7.3f} {fp / (fp + tn):>7.3f}")

print("\nDemographic parity compares the first column; equal opportunity the second;")
print("equalised odds the second AND third. They cannot all hold when base rates differ -")
print("that is a theorem, so the choice has to be argued from the harms, not from a library.")

## 6. The model card

Complete this for the model above (and then for your project model - the project requires
one).

```
Model: <name>, <version>, <date>
Intended use:      <the decision it supports, and for whom>
Out of scope:      <uses that would be wrong, explicitly>
Data:              <source, period, n, known gaps, consent basis>
Performance:       overall AUC ___ [CI ___]; group A ___ [CI ___]; group B ___ [CI ___]
Threshold:         ___ , chosen because ___
Known limitations: <the failure modes you found above>
Monitoring:        <what signals drift, checked how often, by whom>
```

## Exercises

1. **Two thresholds?** Choose a threshold per group so that recall is equal. What does that
   cost in precision, and would you defend it to the people affected?
2. **How much data would fix group B?** Retrain on a training set with group B upsampled.
   Does the gap close, and does anything get worse?
3. **Write the card.** Fill in the block above for this model in full sentences. A card with
   an honest "should not currently be used for group B" line is a better deliverable than a
   confident one that omits section 3.